# 10.0 - Results

Collects every continual-learning arm that has been run (notebooks 4.1 to 4.6), reads
each `arm_*.json` from Drive, and answers the three questions the study exists to answer:

1. Does Naive collapse? It must, or forgetting is not happening.
2. Does the paid method (replay, ~2 MB/class) beat the free ones (EWC, LwF, 0 bytes)?
3. How far is the best method from the joint-probe ceiling from notebook 2.0?

Every number is read against that ceiling. "71%" alone means nothing; "71% against an
82% ceiling" is a sentence.

## **Connect Colab**

In [ ]:
import os, sys, subprocess
from getpass import getpass

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "embeddings"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

## **Download DATASET**

Only Drive is needed here - the results JSONs live on Drive. No dataset download.

In [ ]:
from src.bootstrap import setup
setup(drive=True, dataset=False)

In [ ]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

## **Load Results**

Read the ceiling from notebook 2.0, then every arm JSON notebooks 4.1 to 4.6 wrote. Any
arm that has not been run yet is simply absent - the table shows what exists.

In [ ]:
CEILING = None
diag_path = f"{cfg.RESULTS_DIR}/stage2_embedding_diagnostics.json"
if os.path.exists(diag_path):
    with open(diag_path) as f:
        CEILING = json.load(f)["ceiling"]

print(f"ceiling (2.0 joint linear probe): {CEILING:.2%}" if CEILING else "no ceiling json yet")

In [ ]:
arm_paths = sorted(glob(f"{cfg.RESULTS_DIR}/stage1_arm_*.json"))

ARMS = []
for p in arm_paths:
    with open(p) as f:
        ARMS.append(json.load(f))

print(f"arms found: {len(ARMS)}")
for a in ARMS:
    print(f"  {a['arm']}")

Stop here if nothing was found: run notebooks 4.1 to 4.6 first, they write these JSONs.

## **Master Table**

One row per method. AA is average accuracy over all tasks after the final task, BWT is
backward transfer (negative means forgetting), forgetting is the drop from each task's
best, base_final is base-class accuracy at the end, and memory is bytes stored per class.
The gap column is how far each method sits below the ceiling.

In [ ]:
rows = []
for a in ARMS:
    m = a["metrics"]
    kb = a.get("bytes_per_class", 0) / 1e3
    gap = (CEILING - m["AA"]) if CEILING is not None else float("nan")
    rows.append({
        "method":     a["arm"],
        "AA":         m["AA"],
        "BWT":        m["BWT"],
        "forgetting": m["forgetting"],
        "base_final": m["base_acc_final"],
        "KB_class":   kb,
        "gap_to_ceil": gap,
    })

results_df = pd.DataFrame(rows).sort_values("AA", ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

In [ ]:
if CEILING is not None:
    best = results_df.iloc[0]
    print(f"ceiling         : {CEILING:.2%}")
    print(f"best method     : {best['method']}  AA={best['AA']:.2%}  (gap {CEILING-best['AA']:.2%})")
    free = results_df[results_df["KB_class"] == 0]
    paid = results_df[results_df["KB_class"] > 0]
    if len(free) and len(paid):
        print(f"best free (0 KB): {free.iloc[0]['method']}  {free.iloc[0]['AA']:.2%}")
        print(f"best paid       : {paid.iloc[0]['method']}  {paid.iloc[0]['AA']:.2%}  "
              f"({paid.iloc[0]['KB_class']:.0f} KB/class)")

## **Accuracy vs Memory**

The money figure. AA on the y-axis, bytes-per-class on a log x-axis. The ceiling is the
dashed line every method is trying to reach; Naive (0 bytes) is the floor. A method only
earns its memory if it sits clearly above the free methods at its left.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for _, r in results_df.iterrows():
    x = max(r["KB_class"], 0.5)   # 0-byte methods pinned to the left edge on log-x
    ax.scatter(x, r["AA"], s=90)
    ax.annotate(r["method"], (x, r["AA"]), textcoords="offset points", xytext=(8, 4), fontsize=10)

if CEILING is not None:
    ax.axhline(CEILING, ls="--", color="gray")
    ax.annotate(f"ceiling {CEILING:.1%}", (0.5, CEILING), textcoords="offset points",
                xytext=(4, 6), fontsize=10, color="gray")

ax.set_xscale("log")
ax.set_xlabel("memory per class (KB, log scale)  -  0-byte methods pinned left")
ax.set_ylabel("Average Accuracy")
ax.set_title("Accuracy vs Memory")
ax.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{cfg.RESULTS_DIR}/results_accuracy_vs_memory.png", dpi=150, bbox_inches="tight")
plt.show()

## **Forgetting**

How much each method forgot. Naive should be the worst; the whole point of the others is
to push this bar down.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

fdf = results_df.sort_values("forgetting", ascending=False)
ax.bar(fdf["method"], fdf["forgetting"] * 100)
ax.set_ylabel("Forgetting (%)")
ax.set_title("Forgetting by method (higher = worse)")
ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(f"{cfg.RESULTS_DIR}/results_forgetting.png", dpi=150, bbox_inches="tight")
plt.show()

## **Per-task Accuracy**

The final row of each method's accuracy matrix: how well every task survived to the end.
Task1 (the makeup classes) is expected to suffer most, exactly as the centroid diagnostic
in notebook 2.0 predicted.

In [ ]:
TASK_NAMES = ARMS[0]["task_names"]
n_task = len(TASK_NAMES)

fig, ax = plt.subplots(figsize=(10, 5))
width = 0.8 / max(len(ARMS), 1)

for k, a in enumerate(ARMS):
    R = np.array(a["matrix"], dtype=float)
    final = R[-1]
    xs = np.arange(n_task) + k * width
    ax.bar(xs, final * 100, width=width, label=a["arm"])

ax.set_xticks(np.arange(n_task) + width * (len(ARMS) - 1) / 2)
ax.set_xticklabels(TASK_NAMES)
ax.set_ylabel("Final accuracy (%)")
ax.set_title("Per-task accuracy after the last task")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{cfg.RESULTS_DIR}/results_per_task.png", dpi=150, bbox_inches="tight")
plt.show()

## **Full CL Report per method**

The accuracy matrix and every metric, printed for each arm.

In [ ]:
for a in ARMS:
    R = np.array(a["matrix"], dtype=float)
    cl_report(R, method_name=a["arm"], class_names_per_task=a["task_names"])

## **Save**

One combined JSON so the principal notebook can quote the comparison without recomputing.

In [ ]:
final = {
    "ceiling":  CEILING,
    "methods":  results_df.to_dict(orient="records"),
    "task_names": ARMS[0]["task_names"] if ARMS else [],
}

os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
with open(f"{cfg.RESULTS_DIR}/final_comparison.json", "w") as f:
    json.dump(final, f, indent=2, default=float)

print(f"Saved -> {cfg.RESULTS_DIR}/final_comparison.json")